<a href="https://colab.research.google.com/github/phoudsavanhKongmany/hoc-big-data/blob/main/lab_03022026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**PYSPARK ESSENTIALS - PHÂN TÍCH DỮ LIỆU BÁN LẺ**

**Mục tiêu**: Sinh viên hiểu và vận dụng được các nhóm hàm cốt lõi của Spark: Inspection, Transformation, Aggregation và I/O. Bối cảnh: Bạn là Data Engineer cho chuỗi siêu thị "TechMart". Bạn cần xử lý dữ liệu giao dịch để báo cáo doanh thu

##**PHẦN 0: CHUẨN BỊ MÔI TRƯỜNG & DỮ LIỆU**
Chạy cell này để cài đặt Spark và tạo dữ liệu giả lập (Mock Data)

In [ ]:
# 1. Cài đặt PySpark (Chạy trên Google Colab)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install -q pyspark

# 2. Khởi tạo Spark Session
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import random

spark = SparkSession.builder.appName("Lab_PySpark_Functions").getOrCreate()


# 3. Tạo dữ liệu giả lập (DataFrame)
data = []
cities = ["Hanoi", "HCM", "Danang", "Cantho"]
products = ["Laptop", "Mouse", "Keyboard", "Headphone", "Monitor"]
categories = ["Computer", "Accessory", "Accessory", "Audio", "Display"]

for i in range(1, 101): # Tạo 100 dòng
    prod_idx = random.randint(0, 4)
    row = (
        f"TRX_{i:03d}",                 # TransactionID
        products[prod_idx],             # Product
        categories[prod_idx],           # Category
        random.randint(10, 50) * 10,    # Price ($100 - $500)
        random.randint(1, 5),           # Quantity
        random.choice(cities)           # City
    )
    data.append(row)

columns = ["TransactionID", "Product", "Category", "Price", "Quantity", "City"]
df = spark.createDataFrame(data, columns)

print("✅ Đã tạo xong dữ liệu mẫu!")

✅ Đã tạo xong dữ liệu mẫu!


##**PHẦN 1: KHÁM PHÁ DỮ LIỆU (INSPECTION)**

**Các hàm**: show(), printSchema(), select(), describe()
###**1. Ví dụ minh họa**
Để hiểu dữ liệu đang có gì, chúng ta cần xem cấu trúc và một vài dòng mẫu.

In [ ]:
# Xem cấu trúc dữ liệu (Tên cột, kiểu dữ liệu)
print("--- Schema ---")
df.printSchema()

# Xem 5 dòng đầu tiên
print("--- 5 Dòng đầu ---")
df.show(5)

# Thống kê mô tả (Count, Mean, Min, Max) cột Giá
print("--- Thống kê giá ---")
df.describe("Price").show()

--- Schema ---
root
 |-- TransactionID: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Price: long (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- City: string (nullable = true)

--- 5 Dòng đầu ---
+-------------+---------+---------+-----+--------+------+
|TransactionID|  Product| Category|Price|Quantity|  City|
+-------------+---------+---------+-----+--------+------+
|      TRX_001|   Laptop| Computer|  370|       2| Hanoi|
|      TRX_002|  Monitor|  Display|  320|       3| Hanoi|
|      TRX_003|   Laptop| Computer|  430|       1|Cantho|
|      TRX_004|Headphone|    Audio|  370|       3|   HCM|
|      TRX_005|    Mouse|Accessory|  370|       1|Cantho|
+-------------+---------+---------+-----+--------+------+
only showing top 5 rows
--- Thống kê giá ---
+-------+------------------+
|summary|             Price|
+-------+------------------+
|  count|               100|
|   mean|             306.4|
| stddev|117.10

###**Bài tập 1 (Sinh viên code tại đây)**
**Yêu cầu:** Hãy dùng hàm select() để chỉ hiển thị 2 cột là Product và City, và chỉ hiện 10 dòng đầu tiên.

In [ ]:
# CODE CỦA BẠN Ở ĐÂY:
# Goi y: df.select(..., ...).show(...)
df.select("product","City").show(10)

+---------+------+
|  product|  City|
+---------+------+
|   Laptop| Hanoi|
|  Monitor| Hanoi|
|   Laptop|Cantho|
|Headphone|   HCM|
|    Mouse|Cantho|
|  Monitor| Hanoi|
|   Laptop|Danang|
| Keyboard|   HCM|
|    Mouse|Cantho|
|Headphone|Cantho|
+---------+------+
only showing top 10 rows


##**PHẦN 2: LỌC DỮ LIỆU (FILTERING)**

**Các hàm:** filter() (hoặc where), toán tử & (AND), | (OR).

###**2. Ví dụ minh họa**
Yêu cầu tìm các đơn hàng bán tại "Hanoi" có giá trị sản phẩm trên $300.

In [ ]:
# Lọc theo 2 điều kiện kết hợp
# Lưu ý: Mỗi điều kiện phải để trong ngoặc đơn ()
high_value_hanoi = df.filter((col("City") == "Hanoi") & (col("Price") > 300))

high_value_hanoi.show()

+-------------+---------+---------+-----+--------+-----+
|TransactionID|  Product| Category|Price|Quantity| City|
+-------------+---------+---------+-----+--------+-----+
|      TRX_001|   Laptop| Computer|  370|       2|Hanoi|
|      TRX_002|  Monitor|  Display|  320|       3|Hanoi|
|      TRX_006|  Monitor|  Display|  340|       3|Hanoi|
|      TRX_013|Headphone|    Audio|  390|       5|Hanoi|
|      TRX_032|    Mouse|Accessory|  310|       3|Hanoi|
|      TRX_033|  Monitor|  Display|  430|       4|Hanoi|
|      TRX_041| Keyboard|Accessory|  400|       1|Hanoi|
|      TRX_044|    Mouse|Accessory|  340|       3|Hanoi|
|      TRX_059|    Mouse|Accessory|  430|       2|Hanoi|
|      TRX_060|   Laptop| Computer|  360|       2|Hanoi|
|      TRX_072|    Mouse|Accessory|  420|       5|Hanoi|
|      TRX_080|    Mouse|Accessory|  410|       5|Hanoi|
|      TRX_083| Keyboard|Accessory|  370|       4|Hanoi|
|      TRX_085|   Laptop| Computer|  480|       3|Hanoi|
|      TRX_086|Headphone|    Au

###**Bài tập 2 (Sinh viên code tại đây)**

**Yêu cầu:** Hãy lọc ra các đơn hàng thuộc danh mục (Category) là "Accessory" HOẶC đơn hàng có số lượng (Quantity) lớn hơn 3.

In [ ]:
# CODE CỦA BẠN Ở ĐÂY:
value_loc_ra_don_hang = df.filter((col("Category") == "Accessory") & (col("Quantity") > 3))
value_loc_ra_don_hang.show()

+-------------+--------+---------+-----+--------+------+
|TransactionID| Product| Category|Price|Quantity|  City|
+-------------+--------+---------+-----+--------+------+
|      TRX_009|   Mouse|Accessory|  200|       4|Cantho|
|      TRX_012|Keyboard|Accessory|  120|       5|Danang|
|      TRX_022|   Mouse|Accessory|  260|       4|Cantho|
|      TRX_026|Keyboard|Accessory|  330|       4|   HCM|
|      TRX_029|Keyboard|Accessory|  330|       4|Danang|
|      TRX_035|Keyboard|Accessory|  350|       5|Danang|
|      TRX_037|Keyboard|Accessory|  220|       5|   HCM|
|      TRX_045|Keyboard|Accessory|  110|       4|Cantho|
|      TRX_049|   Mouse|Accessory|  300|       5| Hanoi|
|      TRX_052|Keyboard|Accessory|  100|       4|   HCM|
|      TRX_058|   Mouse|Accessory|  290|       4|Danang|
|      TRX_064|   Mouse|Accessory|  230|       5|   HCM|
|      TRX_066|Keyboard|Accessory|  400|       5|Danang|
|      TRX_069|Keyboard|Accessory|  500|       4|   HCM|
|      TRX_070|Keyboard|Accesso

##**PHẦN 3: BIẾN ĐỔI CỘT (TRANSFORMATION)**
**Các hàm:** withColumn() (Thêm/Sửa cột), drop() (Xóa cột)
###**3. Ví dụ minh họa**
Chúng ta cần tính tổng tiền cho mỗi đơn hàng. Công thức: Total = Price * Quantity.

In [ ]:
# Tạo cột mới tên là "TotalValue"
df_processed = df.withColumn("TotalValue", col("Price") * col("Quantity"))

df_processed.show(5)

+-------------+---------+---------+-----+--------+------+----------+
|TransactionID|  Product| Category|Price|Quantity|  City|TotalValue|
+-------------+---------+---------+-----+--------+------+----------+
|      TRX_001|   Laptop| Computer|  370|       2| Hanoi|       740|
|      TRX_002|  Monitor|  Display|  320|       3| Hanoi|       960|
|      TRX_003|   Laptop| Computer|  430|       1|Cantho|       430|
|      TRX_004|Headphone|    Audio|  370|       3|   HCM|      1110|
|      TRX_005|    Mouse|Accessory|  370|       1|Cantho|       370|
+-------------+---------+---------+-----+--------+------+----------+
only showing top 5 rows


###**Bài tập 3 (Sinh viên code tại đây)**
**Yêu cầu**: Nhân dịp khuyến mãi, hãy tạo thêm một cột mới tên là DiscountedPrice. Giá trị cột này bằng 90% giá gốc (Price * 0.9).

In [ ]:
# CODE CỦA BẠN Ở ĐÂY:
df_processed_add_new = df.withColumn(" DiscountedPrice" , col ("Price") * 0.9)
df_processed_add_new.show()

+-------------+---------+---------+-----+--------+------+----------------+
|TransactionID|  Product| Category|Price|Quantity|  City| DiscountedPrice|
+-------------+---------+---------+-----+--------+------+----------------+
|      TRX_001|   Laptop| Computer|  370|       2| Hanoi|           333.0|
|      TRX_002|  Monitor|  Display|  320|       3| Hanoi|           288.0|
|      TRX_003|   Laptop| Computer|  430|       1|Cantho|           387.0|
|      TRX_004|Headphone|    Audio|  370|       3|   HCM|           333.0|
|      TRX_005|    Mouse|Accessory|  370|       1|Cantho|           333.0|
|      TRX_006|  Monitor|  Display|  340|       3| Hanoi|           306.0|
|      TRX_007|   Laptop| Computer|  350|       3|Danang|           315.0|
|      TRX_008| Keyboard|Accessory|  340|       3|   HCM|           306.0|
|      TRX_009|    Mouse|Accessory|  200|       4|Cantho|           180.0|
|      TRX_010|Headphone|    Audio|  490|       2|Cantho|           441.0|
|      TRX_011|   Laptop|

##**PHẦN 4: TỔNG HỢP & THỐNG KÊ (AGGREGATION) - QUAN TRỌNG**
**Các hàm**: groupBy(), agg(), sum(), count(), avg(), max().

###**4. Ví dụ minh họa**
Tính tổng doanh thu theo từng Thành phố để xem nơi nào bán tốt nhất.

In [ ]:
# Bước 1: Phải có cột TotalValue trước (lấy từ phần trước)
df_cal = df.withColumn("TotalValue", col("Price") * col("Quantity"))

# Bước 2: GroupBy và Sum
report_city = df_cal.groupBy("City") \
                  .agg(sum("TotalValue").alias("Revenue")) \
                  .orderBy(col("Revenue").desc()) # Sắp xếp giảm dần

report_city.show()

+------+-------+
|  City|Revenue|
+------+-------+
| Hanoi|  27930|
|Danang|  26020|
|   HCM|  23330|
|Cantho|  18100|
+------+-------+



###**Bài tập 4 (Sinh viên code tại đây)**
**Yêu cầu**: Hãy tính Tổng số lượng sản phẩm (Quantity) đã bán được theo từng Danh mục (Category).

In [ ]:
# CODE CỦA BẠN Ở ĐÂY:
# Goi y: groupBy("Category").agg(sum("..."))
report_category = df.groupBy("Category") \
    .agg(sum("Quantity").alias("TotalQuantity")) \
    .orderBy(col("TotalQuantity").desc())

report_category.show()

+---------+-------------+
| Category|TotalQuantity|
+---------+-------------+
|Accessory|          163|
|  Display|           61|
|    Audio|           54|
| Computer|           38|
+---------+-------------+



##**PHẦN 5: SQL TRONG SPARK (SPARK SQL)**
**Khái niệm**: Nếu bạn quen dùng SQL, Spark cho phép bạn viết query trực tiếp.

###**5. Ví dụ minh họa**
Tìm các sản phẩm có giá > 400 bằng câu lệnh SQL.

In [ ]:
# Bước 1: Tạo một "View" tạm thời (giống như bảng ảo)
df.createOrReplaceTempView("SalesTable")

# Bước 2: Viết SQL
sql_result = spark.sql("SELECT Product, Price, City FROM SalesTable WHERE Price > 400")

sql_result.show()

+---------+-----+------+
|  Product|Price|  City|
+---------+-----+------+
|   Laptop|  460|   HCM|
|   Laptop|  470| Hanoi|
|    Mouse|  470|Danang|
|   Laptop|  430|   HCM|
|  Monitor|  480|   HCM|
| Keyboard|  470|   HCM|
|    Mouse|  430|Danang|
|Headphone|  440|Danang|
| Keyboard|  420| Hanoi|
| Keyboard|  470|Cantho|
|  Monitor|  460|   HCM|
|   Laptop|  420|Danang|
|  Monitor|  470|Cantho|
|  Monitor|  440| Hanoi|
|  Monitor|  450|   HCM|
|   Laptop|  410|Cantho|
|  Monitor|  440|Cantho|
| Keyboard|  410|Cantho|
|   Laptop|  420|Cantho|
|   Laptop|  470|Danang|
+---------+-----+------+
only showing top 20 rows


###**Bài tập 5 (Sinh viên code tại đây)**
**Yêu cầu**: Dùng spark.sql để tính trung bình cộng giá (AVG(Price)) của các sản phẩm bán tại 'HCM'.

In [ ]:
# CODE CỦA BẠN Ở ĐÂY:
# Query mau: "SELECT AVG(...) as AvgPrice FROM SalesTable WHERE ..."
df.createOrReplaceTempView("SalesTable")
avg_price_hcm = spark.sql("""
    SELECT
        AVG(Price) AS AvgPrice
    FROM SalesTable
    WHERE City = 'HCM'
""")

avg_price_hcm.show()


+-----------------+
|         AvgPrice|
+-----------------+
|315.2173913043478|
+-----------------+



##**PHẦN KẾT: LƯU TRỮ DỮ LIỆU (PARQUET)**
Sau khi xử lý xong, Data Engineer phải lưu dữ liệu lại.

In [ ]:
# Lưu kết quả ra file Parquet (định dạng tối ưu cho Big Data)
output_path = "processed_sales_data"

# mode("overwrite"): Ghi đè nếu folder đã tồn tại
df.write.mode("overwrite").parquet(output_path)

print(f"✅ Đã lưu dữ liệu vào thư mục: {output_path}")

# Kiểm tra lại bằng cách đọc lên
spark.read.parquet(output_path).show(3)

✅ Đã lưu dữ liệu vào thư mục: processed_sales_data
+-------------+-------+--------+-----+--------+------+
|TransactionID|Product|Category|Price|Quantity|  City|
+-------------+-------+--------+-----+--------+------+
|      TRX_051|Monitor| Display|  440|       1| Hanoi|
|      TRX_052|Monitor| Display|  450|       1|   HCM|
|      TRX_053|Monitor| Display|  290|       3|Danang|
+-------------+-------+--------+-----+--------+------+
only showing top 3 rows
